# Q3 — Feature Engineering and Regression Pipeline

## 1. Date Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv("../data/q3_retail_promotions.csv", parse_dates=["transaction_date"])
print("Shape of the dataset:", data.shape)
data.head()

Shape of the dataset: (1200, 9)


,transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold
0,2022-01-01,28,small,semi-urban,free_gift,1,0,5,224
1,2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348
2,2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249
3,2022-01-02,17,small,urban,free_gift,1,0,7,259
4,2022-01-03,50,medium,semi-urban,bogo,0,0,3,277


In [2]:
data["year"] = data["transaction_date"].dt.year
data["month"] = data["transaction_date"].dt.month
data["day"] = data["transaction_date"].dt.day
data["day_of_week"] = data["transaction_date"].dt.dayofweek
data["is_month_end"] = (data["transaction_date"].dt.day >= 25).astype(int)

data[["transaction_date", "year", "month", "day_of_week", "is_month_end"]].head()

,transaction_date,year,month,day_of_week,is_month_end
0,2022-01-01,2022,1,5,0
1,2022-01-01,2022,1,5,0
2,2022-01-02,2022,1,6,0
3,2022-01-02,2022,1,6,0
4,2022-01-03,2022,1,0,0


## 2. Temporal Train-Test Split

In [3]:
data = data.sort_values("transaction_date").reset_index(drop=True)

split_id = int(len(data) * 0.8)
train_data = data.iloc[:split_id]
test_data = data.iloc[split_id:]

print(f"Train: {len(train_data)} rows")
print(f"Test: {len(test_data)} rows")

Train: 960 rows
Test: 240 rows


***Why a Random Split is inappropriate for time-ordered data: ***
With a random 80/20 split of time-ordered data, it is highly possible that future dates end up in the training data set whereas past dates enter the test data set. What this in turn does is that it will train the model on observations that happen after what the model is expected to predict on. This is not realistic in a real-life scenario. Also the model could end up memorizing patterns that exist across adjacent dates in the test data set. This is also not desirable. What is needed is for the model to learn from a training data set that occurs "before" the test data on which the prediction needs to be made, in order to get a true picture of how well the model generalises and predicts forward in time. A time-ordered data split helps us achieve this. 

## 3. Preprocessing Pipeline

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

target = "items_sold"
drop_cols = ["transaction_date", "items_sold"]

categorical_cols = ["promotion_type", "location_type", "store_size"]
numerical_cols = ["store_id", "is_weekend", "is_festival", "competition_density", "year", "month", "day", "day_of_week", "is_month_end"]

print("Numerical Features:", numerical_cols)
print("Categorical Features:", categorical_cols)
print("Dropped Columns:", drop_cols)
print("Target Variable:", target)

preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), numerical_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)], remainder="drop",)

X_train = train_data.drop(columns=drop_cols)
Y_train = train_data[target]
X_test = test_data.drop(columns=drop_cols)
Y_test = test_data[target]

print("X_train:", X_train.shape, " X_test:", X_test.shape)

Numerical Features: ['store_id', 'is_weekend', 'is_festival', 'competition_density', 'year', 'month', 'day', 'day_of_week', 'is_month_end']
Categorical Features: ['promotion_type', 'location_type', 'store_size']
Dropped Columns: ['transaction_date', 'items_sold']
Target Variable: items_sold
X_train: (960, 12)  X_test: (240, 12)


## 4. Model Training & Evaluation